# 예제 01. 사전학습 모델 불러오기
빅데이터프로그래밍 · 10주차

## 목표
- ResNet18을 불러와 구조를 살펴본다
- 특징 추출 부분과 분류 부분을 구분한다
- 학습된 필터가 무엇을 찾는지 본다

내부 수학보다 **모델을 불러오고 입력·출력을 연결하는 흐름**에 집중합니다.

**런타임 > 런타임 유형 변경 > T4 GPU** 를 먼저 선택하세요.


In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. 불러오기
`weights` 를 지정하면 ImageNet 120만 장으로 학습된 가중치가 함께 옵니다.


In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

print("파라미터:", f"{sum(p.numel() for p in model.parameters()):,}개")
print("학습 가능:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}개")


In [ ]:
# 가중치 없이 껍데기만 가져오면 — 전이학습이 아닙니다
blank = models.resnet18(weights=None)
print("사전학습 첫 필터 평균:", model.conv1.weight.data.mean().item())
print("무작위   첫 필터 평균:", blank.conv1.weight.data.mean().item())


## 2. 구조를 크게 세 부분으로 보기


In [ ]:
for name, child in model.named_children():
    n = sum(p.numel() for p in child.parameters())
    print(f"{name:10s} {child.__class__.__name__:18s} {n:>12,}개")


| 구역 | 하는 일 |
| --- | --- |
| `conv1` ~ `layer4` | 특징 추출 — 선, 모양, 질감, 부분 |
| `avgpool` | 공간을 하나로 요약 |
| `fc` | 분류 — ImageNet 1,000개 클래스 |

우리가 바꿀 곳은 마지막 `fc` 하나입니다.


In [ ]:
print("마지막 계층:", model.fc)
print("\n입력 특징 수 :", model.fc.in_features)
print("출력 클래스 수:", model.fc.out_features, "← ImageNet 1,000개")


## 3. 층을 지날 때 shape이 어떻게 변하는가
8주차에서 했던 shape 추적을 큰 모델에서 합니다.


In [ ]:
x = torch.randn(2, 3, 224, 224)      # 컬러 3채널, 224×224

h = x
print(f"{'입력':12s} {tuple(h.shape)}")
for name, child in model.named_children():
    if name == "fc":
        h = torch.flatten(h, 1)
        print(f"{'flatten':12s} {tuple(h.shape)}")
    h = child(h)
    print(f"{name:12s} {tuple(h.shape)}")


`layer4` 를 지나면 (2, 512, 7, 7) 입니다. `avgpool` 이 7×7을 1×1로 줄이고, 512개 숫자가 `fc` 로 들어갑니다.


## 4. 학습된 첫 필터 보기
9주차에서 직접 학습시킨 필터와 비교해 보세요. 훨씬 뚜렷합니다.


In [ ]:
w = model.conv1.weight.data.clone()
w = (w - w.min()) / (w.max() - w.min())          # 0~1로 정규화

fig, axes = plt.subplots(4, 16, figsize=(16, 4.4))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(w[i].permute(1, 2, 0))              # (3,7,7) → (7,7,3)
    ax.axis("off")
plt.suptitle("ResNet18 conv1 — 64개 필터 (7x7, 컬러)", y=1.02)
plt.tight_layout(); plt.show()


방향이 다른 경계선과 색 대비 패턴이 보입니다. 이 필터들은 어떤 이미지에서든 쓸 수 있습니다.


## 5. 다른 모델도 같은 방식입니다


In [ ]:
import pandas as pd

rows = []
for name, fn, weights in [
    ("resnet18",   models.resnet18,   models.ResNet18_Weights.DEFAULT),
    ("resnet50",   models.resnet50,   models.ResNet50_Weights.DEFAULT),
    ("mobilenet_v2", models.mobilenet_v2, models.MobileNet_V2_Weights.DEFAULT),
]:
    m = fn(weights=weights)
    last = m.fc if hasattr(m, "fc") else m.classifier[-1]
    rows.append({"모델": name,
                 "파라미터": f"{sum(p.numel() for p in m.parameters()):,}",
                 "마지막 계층": last.__class__.__name__,
                 "in_features": last.in_features})
pd.DataFrame(rows)


ResNet 계열은 `fc`, MobileNet은 `classifier[-1]` 입니다. 이름만 다르고 방법은 같습니다.


## 직접 해보기
1. `models.resnet34` 를 불러와 파라미터 수와 `fc.in_features` 를 확인하세요.
2. `layer1` 의 내부 구조를 출력해 보세요. (`print(model.layer1)`)


In [ ]:
# 여기에 작성하세요
